# EventConsole Demo

This notebook showcases the `EventConsole` — a rich-based display for grasp-agents event streams.
It works in both terminals and Jupyter notebooks.

**Examples covered:**
1. Agents spawning subagents dynamically (`AgentTool`)
2. Nested sequential workflows
3. Workflows as tools
4. Human-in-the-loop with `AskUserTool`
5. Multi-agent Runner with dynamic routing

In [1]:
from typing import Any
from collections.abc import Sequence

from dotenv import load_dotenv
from pydantic import BaseModel

from grasp_agents import (
    LLMAgent,
    RunContext,
    BaseTool,
    function_tool,
)
from grasp_agents.agent.agent_tool import AgentTool
from grasp_agents.console import EventConsole, stream_events
from grasp_agents.llm_providers.litellm import LiteLLM, LiteLLMSettings
from grasp_agents.llm_providers.openai_responses import (
    OpenAIResponsesLLM,
    OpenAIResponsesLLMSettings,
)
from openai.types.shared import Reasoning


from grasp_agents.runner.runner import Runner, END_PROC_NAME
from grasp_agents.types.events import ProcPacketOutEvent, RunPacketOutEvent
from grasp_agents.workflow.sequential_workflow import SequentialWorkflow

load_dotenv()

True

## Observability with Phoenix (optional)

Start Phoenix locally:
```bash
cd ./phoenix
docker compose up -d
export TELEMETRY_COLLECTOR_HTTP_ENDPOINT=http://localhost:6006/v1/traces
```

All agent spans will appear in the Phoenix UI at http://localhost:6006.

In [2]:
from grasp_agents.telemetry import init_tracing
from grasp_agents.telemetry.phoenix import init_phoenix

init_tracing(project_name="console-demo")
init_phoenix(project_name="console-demo")

## Shared LLM setup

In [3]:
# llm = LiteLLM(
#     model_name="anthropic/claude-sonnet-4-6",
#     llm_settings=LiteLLMSettings(
#         max_completion_tokens=4000,
#         thinking={"type": "enabled", "budget_tokens": 2000},
#     ),
# )

llm = OpenAIResponsesLLM(
    model_name="gpt-5.4",
    llm_settings=OpenAIResponsesLLMSettings(
        reasoning=Reasoning(effort="medium", summary="detailed")
    ),
)

---
## 1. Agents spawning subagents (`AgentTool`)

An `AgentTool` spawns a fresh LLM agent per invocation. The **coordinator**
controls the subagent's behavior through the prompt it passes — no need for
separate tool definitions for different roles.


In [4]:
# A single general-purpose subagent — the coordinator provides the role via prompt
subagent = AgentTool[None](
    name="subagent",
    description="A general-purpose assistant. Provide a clear task in the prompt.",
    llm=llm,
    max_turns=1,
)

coordinator = LLMAgent[str, str, None](
    name="coordinator",
    llm=llm,
    tools=[subagent],
    sys_prompt=(
        "You are a content coordinator. When given a topic:"
        "1. Use the subagent tool to research the topic — tell it to return 3 concise bullet points"
        "2. Use the subagent tool again to write a polished paragraph from those findings"
        "Return the final paragraph as your answer."
    ),
    stream_llm_responses=True,
)

In [5]:
ctx = RunContext[None]()

async for event in stream_events(
    coordinator.run_stream("The history of the Internet", ctx=ctx),
    show_thinking=True,
    show_input_messages=True,
):
    pass

╭─ System → coordinator ───────────────────────────────────────────────────────╮
│ You are a content coordinator. When given a topic:1. Use the subagent tool   │
│ to research the topic — tell it to return 3 concise bullet points2. Use the  │
│ subagent tool again to write a polished paragraph from those findingsReturn  │
│ the final paragraph as your answer.                                          │
╰──────────────────────────────────────────────────────────────────────────────╯

╭─ User → coordinator ─────────────────────────────────────────────────────────╮
│ The history of the Internet                                                  │
╰──────────────────────────────────────────────────────────────────────────────╯

───────────────────────────── coordinator · turn 1 ─────────────────────────────

  ┌ thinking
  │ 
  │ **Planning the research process**
  │ 
  │ I’m realizing that I need to follow the developer's instructions carefully.
  │ I should use the subagent tool twice: firs

---
## 2. Nested sequential workflow

A `SequentialWorkflow` chains processors: the output of each becomes the input of the next.

Here we build a two-stage **analysis pipeline**:
1. `analyst` — lists pros and cons
2. `summarizer` — distills them into a balanced conclusion

In [6]:
analyst = LLMAgent[str, str, None](
    name="analyst",
    llm=llm,
    sys_prompt=(
        "You are an analyst. Given a topic, list exactly 3 pros and 3 cons. "
        "Use bullet points. Be concise."
    ),
    stream_llm_responses=True,
)

summarizer = LLMAgent[str, str, None](
    name="summarizer",
    llm=llm,
    sys_prompt=(
        "You are a summarizer. Given a pros/cons analysis, write a single "
        "balanced conclusion paragraph (2-3 sentences). No preamble."
    ),
    stream_llm_responses=True,
)

pipeline = SequentialWorkflow[str, str, None](
    name="analysis_pipeline",
    subprocs=[analyst, summarizer],
)

In [7]:
ctx = RunContext[None]()

async for event in stream_events(
    pipeline.run_stream(chat_inputs="Remote work vs office work", ctx=ctx),
    show_thinking=True,
    show_input_messages=True,
):
    if isinstance(event, ProcPacketOutEvent) and event.source == "analysis_pipeline":
        conclusion = event.data.payloads[0]

# print(f"\nFinal conclusion: {conclusion}")

╭─ System → analyst ───────────────────────────────────────────────────────────╮
│ You are an analyst. Given a topic, list exactly 3 pros and 3 cons. Use       │
│ bullet points. Be concise.                                                   │
╰──────────────────────────────────────────────────────────────────────────────╯

╭─ User → analyst ─────────────────────────────────────────────────────────────╮
│ Remote work vs office work                                                   │
╰──────────────────────────────────────────────────────────────────────────────╯

─────────────────────────────── analyst · turn 1 ───────────────────────────────

  ┌ thinking
  │ 
  │ **Evaluating remote work vs office work**
  │ 
  │ I need to provide exactly three pros and three cons for remote work compared
  │ to office work. It seems like listing concise bullets would be the best way
  │ to present this information clearly. I’ll make sure to frame the pros and
  │ cons effectively, ensuring there are 

---
## 3. Workflows as tools

Any processor (including workflows) can be converted to a tool via `.as_tool()`,
enabling **manager/worker** architectures.

Here a `manager` agent delegates to the analysis pipeline from Example 2.
With `stream_tools=True`, the inner workflow's events stream through the EventConsole.

In [8]:
# Rebuild the pipeline with a BaseModel input (required for tools)


class TopicInput(BaseModel):
    topic: str


analyst_v2 = LLMAgent[TopicInput, str, None](
    name="analyst",
    llm=llm,
    in_prompt="Analyze the following topic — list 3 pros and 3 cons:\n\n{topic}",
    sys_prompt="You are an analyst. Use concise bullet points.",
    stream_llm_responses=True,
)

summarizer_v2 = LLMAgent[str, str, None](
    name="summarizer",
    llm=llm,
    sys_prompt=(
        "Write a single balanced conclusion paragraph (2-3 sentences) "
        "from the analysis. No preamble."
    ),
    stream_llm_responses=True,
)

pipeline_v2 = SequentialWorkflow[TopicInput, str, None](
    name="analysis_pipeline",
    subprocs=[analyst_v2, summarizer_v2],
)

# Convert to tool
analysis_tool = pipeline_v2.as_tool(
    tool_name="deep_analysis",
    tool_description=(
        "Run a deep pros/cons analysis on any topic. Returns a balanced conclusion."
    ),
)

# Manager uses the tool
manager = LLMAgent[str, str, None](
    name="manager",
    llm=llm,
    tools=[analysis_tool],
    sys_prompt=(
        "You are a project manager. Use the deep_analysis tool to "
        "analyze the user's question, then present the conclusion."
    ),
    stream_llm_responses=True,
    stream_tools=True,
)

In [ ]:
ctx = RunContext[None]()

async for event in stream_events(
    manager.run_stream("Should our company adopt a 4-day work week?", ctx=ctx),
    show_thinking=True,
):
    pass

─────────────────────────────── manager · turn 1 ───────────────────────────────

  ┌ thinking
  │ 
  └

╭─ manager → deep_analysis ────────────────────────────────────────────────────╮
│ topic  Should our company adopt a 4-day work week?                           │
╰──────────────────────────────────────────────────────────────────────────────╯


---
## 4. Human-in-the-loop

Agents can involve the user via custom tools. Here an **advisor** agent asks
clarifying questions before making recommendations.

In Jupyter, `input()` creates an interactive text widget.

In [ ]:
class UserQuestion(BaseModel):
    question: str


class AskUserTool(BaseTool[UserQuestion, str, None]):
    name = "ask_user"
    description = "Ask the user a clarifying question and get their response."

    async def _run(self, inp: UserQuestion, **kwargs: Any) -> str:  # type: ignore[override]
        return input(f"\n{inp.question}\n> ")


advisor = LLMAgent[str, str, None](
    name="advisor",
    llm=llm,
    tools=[AskUserTool()],
    sys_prompt=(
        "You are a helpful travel advisor. When the user asks for recommendations:\n"
        "1. Ask 2 clarifying questions (one at a time) using the ask_user tool — "
        "e.g. budget range, interests, travel dates.\n"
        "2. Then provide 2-3 tailored destination recommendations.\n"
        "Keep questions concise."
    ),
    stream_llm_responses=True,
)

In [ ]:
ctx = RunContext[None]()

async for event in stream_events(
    advisor.run_stream("I want to go on vacation next month", ctx=ctx),
    show_thinking=True,
):
    pass

─────────────────────────────── advisor · turn 1 ───────────────────────────────

  ┌ thinking
  │ 
  │ **Clarifying user needs**
  │ 
  │ I see the user is asking for a recommendation, so I need to get more
  │ details. I'll ask two clarifying questions—one at a time, to keep it simple
  │ and focused. The first question should be concise. It’s important to use the
  │ ask_user tool to do this, and I’ll make sure to use the commentary tool for
  │ clarity. This way, I can better understand what the user wants!
  └

╭─ advisor → ask_user ─────────────────────────────────────────────────────────╮
│ question  What’s your approximate budget per person?                         │
╰──────────────────────────────────────────────────────────────────────────────╯

                    gpt-5.4-2026-03-05 · 128 in · 66 out · 39 thinking · $0.0013



---
## 5. Multi-agent Runner with dynamic routing

The `Runner` orchestrates multiple agents connected in a graph.
Each agent declares its possible recipients; a `@add_recipient_selector`
hook picks the actual destination at runtime.

Here we build a **proposal review loop**:

```
writer (entry) ──► reviewer ──► writer   (if REVISE)
                            └──► *END*    (if APPROVED)
```

The reviewer either sends the proposal back for revision or approves it.

In [ ]:
proposal_writer = LLMAgent[str, str, None](
    name="writer",
    llm=llm,
    recipients=["reviewer"],
    sys_prompt=(
        "You are a creative writer.\n"
        "- If given a topic: write a bold, creative 2-sentence proposal.\n"
        "- If given revision feedback: improve your proposal.\n"
        "Output only the proposal text, nothing else."
    ),
    stream_llm_responses=True,
)

reviewer = LLMAgent[str, str, None](
    name="reviewer",
    llm=llm,
    recipients=["writer", END_PROC_NAME],
    sys_prompt=(
        "You are a strict but fair reviewer. Evaluate the proposal:\n"
        '- If it\'s compelling, respond starting with "APPROVED:" followed by brief praise.\n'
        '- If it needs improvement, respond starting with "REVISE:" followed by one specific suggestion.\n'
        "Be concise (1-2 sentences). Do not accept the first proposal without revision — push for improvement."
    ),
    stream_llm_responses=True,
)


@reviewer.add_recipient_selector
def route_review(output: str, **kwargs: Any) -> Sequence[str]:
    if output.strip().upper().startswith("APPROVED"):
        return [END_PROC_NAME]
    return ["writer"]

In [ ]:
ctx = RunContext[None]()
runner = Runner(
    entry_proc=proposal_writer,
    procs=[proposal_writer, reviewer],
    ctx=ctx,
    name="proposal_review",
)

async for event in stream_events(
    runner.run_stream(chat_inputs="A new public transit system for mid-size cities")
):
    if isinstance(event, RunPacketOutEvent):
        final = event.data.payloads[0]

─────────────────────────────── writer · turn 1 ────────────────────────────────

Imagine a modular public transit system built for mid-size cities like a living circulatory network: sleek electric trams on main arteries, on-demand neighborhood shuttles feeding every stop, and a single smart pass that makes the whole city feel five minutes smaller. Designed as both infrastructure and identity, it would turn commuting into civic pride by pairing rapid, low-emission mobility with beautiful stations, real-time adaptive routing, and streets reclaimed for people instead of traffic.

                    gpt-5.4-2026-03-05 · 61 in · 124 out · 27 thinking · $0.0020

────────────────────────────── reviewer · turn 1 ───────────────────────────────

REVISE: Add one concrete implementation detail—such as projected ridership gains, cost per mile, or a phased rollout plan—to make the vision feel credible rather than purely aspirational.

         gpt-5.4-2026-03-05 · 173 in · 84 out · 40 thinking · 

---
## 6. Background subagent with concurrent foreground tools

A coordinator agent spawns a **background research subagent** (slow, multi-turn)
while continuing to call quick foreground tools. The `EventConsole` shows both
the background task lifecycle (launch/complete) and the foreground tool calls
interleaved in real time.

Key pattern:
- `@function_tool(background=True)` for the slow research task
- Regular `@function_tool` for quick lookups
- The agent loop stays alive while background tasks are pending

In [4]:
# Background subagent — a real LLM agent that runs as a background tool
researcher = AgentTool[None](
    name="deep_research",
    description=(
        "Run an in-depth research analysis on a given topic. "
        "Return 5 bullet points covering: historical context, "
        "current state, emerging trends, key players, and risks."
    ),
    llm=llm,
    max_turns=1,
    background=True,
)


# Quick foreground tools
@function_tool
async def check_calendar(date: str) -> str:
    """Check calendar availability for a date."""
    return f"{date}: 2 meetings scheduled, 3 hours free"


@function_tool
async def get_team_status(team: str) -> str:
    """Get current status of a team."""
    return f"Team {team}: 4 members active, sprint ends Friday, 2 PRs pending review"


@function_tool
async def send_notification(recipient: str, message: str) -> str:
    """Send a notification to a team member."""
    return f"Notification sent to {recipient}"


planner = LLMAgent[str, str, None](
    name="planner",
    llm=llm,
    tools=[researcher, check_calendar, get_team_status, send_notification],
    sys_prompt=(
        "You are a project planner. When given a project topic:\n"
        "1. Start deep_research on the topic (finishes in the background, you will get a notification when it finishes: DO NOT call it again)\n"
        "2. While research is running, check the calendar for next Monday\n"
        "3. Get the engineering team status\n"
        "4. Send a notification to the team lead about the new project\n"
        "5. When research results arrive, summarize everything into a plan\n"
        "Do steps 2-4 immediately, don't wait for the research to finish."
    ),
    stream_llm_responses=True,
)

In [5]:
ctx = RunContext[None]()

async for event in stream_events(
    planner.run_stream("AI-powered code review tooling", ctx=ctx),
    show_thinking=True,
):
    pass

─────────────────────────────── planner · turn 1 ───────────────────────────────

  ┌ thinking
  │ 
  │ **Planning project execution**
  │ 
  │ I need to act as a planner based on the user's project topic. First, I'll
  │ initiate deep research in the background without needing to call again.
  │ While the research is running, I'll check the calendar for next Monday
  │ (April 20, 2026), get the engineering team's status, and send a notification
  │ to the team lead. After the research results come in, I’ll summarize
  │ everything into a plan, possibly using parallel steps for the next actions
  │ while the research runs.**Deciding on research execution**
  │ 
  │ I'm thinking about whether to parallelize steps 2-4 while waiting for the
  │ research results. The tool should finish in the background and notify me
  │ when it's done, so I shouldn't call it again. However, I wonder if the tool
  │ will return an immediate result or if I'll need to wait. It seems best to
  │ initiate deep

---
## EventConsole options

The `EventConsole` and `stream_events()` accept several options:

| Option | Default | Description |
|---|---|---|
| `show_thinking` | `False` | Display reasoning/thinking blocks |
| `show_tool_args` | `True` | Stream tool call arguments |
| `show_usage` | `True` | Show model name and token counts |
| `show_input_messages` | `False` | Display system/user message events |
| `show_packets` | `False` | Display processor/runner packet output |
| `trunc_len` | `10000` | Max content length before truncation |
| `console` | auto | Custom `rich.Console` instance |

You can also use `EventConsole` directly for more control:

```python
ec = EventConsole(show_thinking=True, show_usage=False)
async for event in ec.stream(agent.run_stream("hello", ctx=ctx)):
    ...
```